# Track V — V7: risk-v2 policy fit on FoR val-B (notebook 10)

This notebook runs the model v2 risk-policy fit (Protocol v1.0 §6 unchanged; D2, D9, D45, D49, D61, D64) once, from one exact reviewed commit, in notebook 05's proven order: it reads the search winner and its bundle seed from the committed search evidence, builds the val-B log-Mel cache twice for a log-Mel winner (`cnn-aug`, `crnn`; `aasist` takes the F3 waveform and builds none), copies the bundle-seed checkpoint and the selection set S from Drive, runs `python -m src.tuning.v2_risk_run` once and exports the policy, the per-clip val-B scores and the reports to Drive ([docs/v2_training.md](../docs/v2_training.md), "v2 risk policy" and "Notebook 10"). The runner checks the checkpoint against the committed search evidence, reproduces its selection metric M exactly on S, then scores val-B and fits the policy.

Only val-A (through S) and val-B are read: train and test are never cached, and the test split is never requested (D9, D31). Scoring runs on CPU by design; the T4 runtime is required only because it is the image that reproduces the Stage 8 and Stage 10 log-Mel bytes (D58 (1), Entry 94). Nothing is logged to an external service; the committed reports are the record (D40). Run it only after the search evidence PR has committed `docs/audit_evidence/v2_search/`.

## 1. Parameters

Select the Colab runtime **T4 GPU** and, under "Version d'exécution / Runtime version", **2026.10** before running (D58 (1): Python 3.13.16, torch 2.11.0+cu130). The GPU is never used for scoring. Replace `COMMIT` with the full 40-character approved commit (a `main` commit that already holds `docs/audit_evidence/v2_search/`). `VERSION` names the risk-policy folder and is passed to the runner. `SEARCH_VERSION` is notebook 08's constant: it names the search folder on Drive and must equal the `version` in the committed search report. There is no winner, seed or path knob: the winner and the bundle seed come from the committed search evidence, the scoring batch size from `configs/v2_candidates.yaml` and the policy fit from `src.tuning.risk_fit`.

In [ ]:
REPO_URL = 'https://github.com/kahlanzzi/voice-spoofing-detection.git'
COMMIT = 'REPLACE_WITH_APPROVED_COMMIT'  # full 40-character SHA
VERSION = 'risk-v2'
SEARCH_VERSION = 'model-v2'  # notebook 08's constant: the search folder and the search reports' version
DRIVE_ROOT = '/content/drive/MyDrive/voice-spoofing-detection'

## 2. Mount Drive; the Track V Drive layout; refuse existing outputs

All paths are constants under `DRIVE_ROOT`, as in the "Drive layout (Track V)" table of `docs/v2_training.md`:

| Constant | Drive path | Here |
|---|---|---|
| `SEARCH_OUT` | `checkpoints/v2_search/<SEARCH_VERSION>/` (`runs/<winner>_seed<k>/checkpoint/`) | read (the bundle-seed checkpoint) |
| `SELECTION_SET_DRIVE` | `processed/v2/selection_set_s/` | read |
| `CHECKPOINT_DEST` | `checkpoints/risk/<VERSION>/` (`risk_policy.json`, `val_b_scores.npz`) | written once |
| `EVIDENCE_DEST` | `processed/metadata/v2_risk/` (`run_report.json`, `diagnostics_report.json`, export manifest) | written once |
| FoR archive | `raw/for_2sec_ff8c82c7.tar` and its revision marker | read |

A frozen policy or evidence folder is never overwritten, so the notebook stops here, before any expensive work, if either output folder already exists. The same check runs again right before export. The v1 folders (`checkpoints/risk/risk-v1/`, `processed/metadata/stage10/`) are never touched.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path(DRIVE_ROOT)
SEARCH_OUT = DRIVE / 'checkpoints' / 'v2_search' / SEARCH_VERSION
SELECTION_SET_DRIVE = DRIVE / 'processed' / 'v2' / 'selection_set_s'
CHECKPOINT_DEST = DRIVE / 'checkpoints' / 'risk' / VERSION
EVIDENCE_DEST = DRIVE / 'processed' / 'metadata' / 'v2_risk'


def refuse_existing_outputs():
    for dest in (CHECKPOINT_DEST, EVIDENCE_DEST):
        if dest.exists():
            raise FileExistsError(f'{dest} already exists; risk-v2 outputs are never overwritten')


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


refuse_existing_outputs()
assert (DRIVE / 'raw').is_dir(), f'{DRIVE / "raw"} not found; check DRIVE_ROOT'

## 3. Check out the exact commit

As notebook 05: the repository is cloned into `/content` and the full commit is checked out detached. HEAD must equal `COMMIT` exactly and the worktree must be clean. `assert_clean` also refuses ignored files other than Python bytecode caches, because `.gitignore` hides some folders from `git status`; it runs before and after every long cell. Every later output (caches, pytest XML, the input copies, the run output) is written outside the clone, so the runner, which refuses a dirty worktree, sees a clean tree. `stream` prints a child's output as it runs; `probe` runs a short script in a fresh process inside the clone and returns its JSON output.

In [ ]:
REPO = Path('/content/voice-spoofing-detection')
assert len(COMMIT) == 40, f'COMMIT must be the full 40-character SHA, got {COMMIT!r}'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=REPO, check=True)


def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout


def assert_clean():
    head = git('rev-parse', 'HEAD').strip()
    assert head == COMMIT, (head, COMMIT)
    status = git('status', '--porcelain')
    assert not status, status
    ignored = [line[3:] for line in git('status', '--porcelain', '--ignored').splitlines() if line.startswith('!! ')]
    unexpected = [path for path in ignored if not path.rstrip('/').endswith('__pycache__')]
    assert not unexpected, unexpected


def stream(command, **kwargs):
    # Like subprocess.run(check=True), but prints the child's output line by line as it runs.
    env = {**kwargs.pop('env', os.environ), 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                          **kwargs) as process:
        for line in process.stdout:
            print(line, end='')
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


def probe(script):
    # Run a short Python script in a fresh process inside the clone; return its JSON output.
    result = subprocess.run([sys.executable, '-c', script], cwd=REPO, capture_output=True, text=True)
    if result.returncode:
        print(result.stderr)
        result.check_returncode()
    return json.loads(result.stdout)


assert_clean()
os.chdir(REPO)
print('HEAD', COMMIT)

## 4. Runtime gate and the D64 version guard

The runtime must be a Colab **T4** with runtime **2026.10**: Python exactly 3.13.16 and Colab's preinstalled torch exactly 2.11.0+cu130 (D58 (1), as notebooks 06 and 08). The T4 is required because it is the runtime that reproduces the Stage 8 and Stage 10 log-Mel bytes (Entry 94: a CPU runtime gave a different val-A `npz`); the GPU is **never used for scoring**: the runner scores on CPU by design and takes no device argument. Torch is never installed: only `requirements.txt` and `requirements-cnn.txt` are installed, and neither lists torch.

Then, in a **fresh process** after the install, Python and torch are read again (a silent upgrade during the install is caught) and `src.augmentation.version_guard.require_versions()` runs: NumPy, SciPy and SoundFile must equal the `requirements.txt` pins (D64). Its result is printed and recorded in the export manifest; any mismatch stops the notebook here. This is the same guard notebook 08 calls; the notebook has no version helper of its own.

In [ ]:
import torch

GPU_NAME = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                          check=True, capture_output=True, text=True).stdout.strip()
assert 'T4' in GPU_NAME, f'D58 (1): a Colab T4 runtime is required, got {GPU_NAME!r}'
assert torch.__version__ == '2.11.0+cu130', f'D58 (1): runtime 2026.10 (torch 2.11.0+cu130) required, got {torch.__version__}'
assert_clean()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-cnn.txt'],
               cwd=REPO, check=True)
ENV = probe('''
import json, platform, numpy, torch
from src.augmentation.version_guard import require_versions
print(json.dumps({
    'python': platform.python_version(),
    'numpy': numpy.__version__,
    'torch': torch.__version__,
    'require_versions': require_versions(),
}))
''')
assert ENV['python'] == '3.13.16', f"D58 (1): Python 3.13.16 required, got {ENV['python']}"
assert ENV['torch'] == '2.11.0+cu130', f"D58 (1): torch 2.11.0+cu130 required, got {ENV['torch']}"
VERSIONS = ENV.pop('require_versions')
ENV['gpu_name'] = GPU_NAME
assert_clean()
print('require_versions:', VERSIONS)
for key, value in ENV.items():
    print(f'{key}: {value}')

## 5. The winner and its bundle seed, from the committed search evidence

The winner, the bundle seed and the expected checkpoint SHA-256 are read with the runner's own rule, `v2_risk_run.bundle_checkpoint(docs/audit_evidence/v2_search)`, in a fresh process (the two search reports must agree; no rule is duplicated here). The committed search `run_report.json` must name `SEARCH_VERSION`, so the checkpoint is taken from the search folder the evidence describes. The winner's input kind decides the val-B input: `logmel` (`cnn-aug`, `crnn`) uses the val-B log-Mel cache built in step 8 (`--cache-dir`); `waveform` (`aasist`) reads val-B through F3 from the extracted FoR root (`--root`, PR V4c). The F3 profile file must match the Stage 6 evidence.

In [ ]:
MANIFEST = REPO / 'data/manifests/for2sec_frozen_manifest.csv'
PROFILE = REPO / 'configs/preprocessing.yaml'
F3 = json.loads((REPO / 'docs/audit_evidence/stage6_preprocessing/profile_report.json').read_text(encoding='utf-8'))['profile']
assert sha256(PROFILE) == F3['config_sha256'], 'configs/preprocessing.yaml differs from F3'
SEARCH_EVIDENCE = REPO / 'docs/audit_evidence/v2_search'
assert (SEARCH_EVIDENCE / 'run_report.json').is_file(), f'{SEARCH_EVIDENCE} is missing: merge the search evidence PR first'
SEARCH_REPORT = json.loads((SEARCH_EVIDENCE / 'run_report.json').read_text(encoding='utf-8'))
assert SEARCH_REPORT['version'] == SEARCH_VERSION, (SEARCH_REPORT['version'], SEARCH_VERSION)
assert_clean()
BUNDLE = probe(f'''
import json
from pathlib import Path
from src.models.v2_checkpoint import INPUTS
from src.tuning.v2_risk_run import bundle_checkpoint
from src.tuning.v2_search import run_dir_name
winner, seed, checkpoint_sha256 = bundle_checkpoint(Path({str(SEARCH_EVIDENCE)!r}))
print(json.dumps({{'winner': winner, 'bundle_seed': seed, 'checkpoint_sha256': checkpoint_sha256,
                  'input_kind': INPUTS[winner], 'run_dir': run_dir_name(winner, seed)}}))
''')
WINNER, BUNDLE_SEED = BUNDLE['winner'], BUNDLE['bundle_seed']
INPUT_KIND, RUN_DIR = BUNDLE['input_kind'], BUNDLE['run_dir']
assert INPUT_KIND in ('logmel', 'waveform'), INPUT_KIND
assert_clean()
print(json.dumps(BUNDLE, indent=2))

## 6. Verify and extract the pinned FoR archive

As notebook 05: the revision marker next to the Drive backup must equal the D21 FoR revision, read here from `configs/audit.yaml`. The archive is extracted with `filter='data'`. Its SHA-256 is computed and written to the export manifest as `for_archive_sha256_observed`: evidence only, never asserted or compared, and not a dataset pin (D21 unchanged). Both branches need it: the val-B log-Mel cache is built from it, and an `aasist` winner reads val-B through F3 from it.

In [ ]:
import tarfile
import yaml

EXPECTED_REVISION = yaml.safe_load((REPO / 'configs/audit.yaml').read_text(encoding='utf-8'))['for2sec']['revision']
ARCHIVE = DRIVE / 'raw' / 'for_2sec_ff8c82c7.tar'
REVISION_FILE = DRIVE / 'raw' / 'for_2sec_REVISION.txt'
assert REVISION_FILE.read_text(encoding='utf-8').strip() == EXPECTED_REVISION
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE = DATA_ROOT / 'for_2sec_ff8c82c7.tar'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
assert_clean()
shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
with tarfile.open(LOCAL_ARCHIVE) as archive:
    archive.extractall(DATA_ROOT, filter='data')
FOR_ARCHIVE_SHA256_OBSERVED = sha256(LOCAL_ARCHIVE)  # recorded only (evidence, not a pin)
assert_clean()
print('FoR archive extracted to', AUDIO_ROOT)
print('for_archive_sha256_observed:', FOR_ARCHIVE_SHA256_OBSERVED)

## 7. Run the test suite: no failure, error or skip

As notebook 05: `REQUIRE_TORCH_TESTS=1` turns a missing torch into a failure. The counts come from the JUnit XML written to `/content` (outside the clone), not from the terminal text, and every failure, error and skip stops the notebook.

In [ ]:
import xml.etree.ElementTree as ET

JUNIT = Path('/content/pytest_v2_risk.xml')
if JUNIT.exists():
    JUNIT.unlink()
assert_clean()
stream([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', f'--junitxml={JUNIT}'],
       cwd=REPO, env={**os.environ, 'REQUIRE_TORCH_TESTS': '1'})
root = ET.parse(JUNIT).getroot()
suites = [root] if root.tag == 'testsuite' else root.findall('testsuite')
COUNTS = {key: sum(int(suite.get(key, 0)) for suite in suites) for key in ('tests', 'failures', 'errors', 'skipped')}
print(COUNTS)
assert COUNTS['tests'] > 0, COUNTS
assert COUNTS['failures'] == COUNTS['errors'] == COUNTS['skipped'] == 0, COUNTS
assert_clean()

## 8. val-B log-Mel cache, built twice and compared (log-Mel winners only)

For `cnn-aug` and `crnn` only (`INPUT_KIND == 'logmel'`); an `aasist` winner builds no cache. val-B is cached into `/content/cache_a` and again into `/content/cache_b`; every sample hash is checked before decoding. Both builds must hold the same files with identical SHA-256s, and the cache must record the F3 profile config SHA-256 and fingerprint (Stage 6 evidence, D39) and `COMMIT`. **Known answer:** its `npz_sha256` must equal the val-B cache hash committed in `docs/audit_evidence/stage10_risk/run_report.json`, the cache risk-v1 was fitted on (the npz only: the JSON holds the code commit). val-B only: S covers val-A, and train and test are never built. The runner reads `cache_a`.

In [ ]:
STAGE10 = json.loads((REPO / 'docs/audit_evidence/stage10_risk/run_report.json').read_text(encoding='utf-8'))
VAL_B_NPZ_EXPECTED = STAGE10['caches']['val-B']['npz_sha256']
CACHE_A, CACHE_B = Path('/content/cache_a'), Path('/content/cache_b')
CACHE_HASHES_A = CACHE_HASHES_B = None
if INPUT_KIND == 'logmel':
    CACHE_SPLITS = ('val-B',)
    for cache in (CACHE_A, CACHE_B):
        if cache.exists():
            shutil.rmtree(cache)
        cache.mkdir()
        for split in CACHE_SPLITS:
            assert_clean()
            subprocess.run([
                sys.executable, '-m', 'src.features.logmel_cache',
                '--root', str(AUDIO_ROOT), '--manifest', str(MANIFEST), '--profile', str(PROFILE),
                '--split', split, '--out', str(cache / split),
            ], cwd=REPO, check=True, stdout=subprocess.DEVNULL)
            assert_clean()

    def file_hashes(folder):
        return {p.relative_to(folder).as_posix(): sha256(p) for p in sorted(folder.rglob('*')) if p.is_file()}

    CACHE_HASHES_A, CACHE_HASHES_B = file_hashes(CACHE_A), file_hashes(CACHE_B)
    assert sorted(CACHE_HASHES_A) == sorted(f'{s}/{s}.{ext}' for s in CACHE_SPLITS for ext in ('npz', 'json')), sorted(CACHE_HASHES_A)
    assert CACHE_HASHES_B == CACHE_HASHES_A, 'the two cache builds differ'
    for split in CACHE_SPLITS:
        meta = json.loads((CACHE_A / split / f'{split}.json').read_text(encoding='utf-8'))
        assert meta['profile_config_sha256'] == F3['config_sha256'], (split, meta['profile_config_sha256'])
        assert meta['profile_fingerprint'] == F3['fingerprint'], (split, meta['profile_fingerprint'])
        assert meta['code_commit'] == COMMIT, (split, meta['code_commit'])
    VAL_B_NPZ = CACHE_HASHES_A['val-B/val-B.npz']
    assert meta['npz_sha256'] == VAL_B_NPZ == VAL_B_NPZ_EXPECTED, f'val-B npz {VAL_B_NPZ}, json {meta["npz_sha256"]}, Stage 10 {VAL_B_NPZ_EXPECTED}'
    print('val-B log-Mel cache equals the Stage 10 known answer:', VAL_B_NPZ)
else:
    print(f'{WINNER} takes the F3 waveform: no log-Mel cache is built; val-B is read through F3 from', AUDIO_ROOT)
assert_clean()

## 9. Copy the bundle-seed checkpoint and S from Drive

The bundle-seed checkpoint folder `checkpoints/v2_search/<SEARCH_VERSION>/runs/<winner>_seed<k>/checkpoint/` (D45: the bundle seed only) and the selection set S (`processed/v2/selection_set_s/`) are copied to `/content/v2_risk_inputs/`, outside the clone. Neither is verified or loaded here: the runner checks the checkpoint bytes against `checkpoint_sha256` in the committed search evidence and its architecture and seed against the winner and bundle seed, and S against the `selection_set_sha256` the checkpoint recorded, before scoring anything.

In [ ]:
INPUTS_LOCAL = Path('/content/v2_risk_inputs')
CHECKPOINT_DRIVE = SEARCH_OUT / 'runs' / RUN_DIR / 'checkpoint'
CHECKPOINT_LOCAL = INPUTS_LOCAL / RUN_DIR / 'checkpoint'
S_LOCAL = INPUTS_LOCAL / 'selection_set_s'
assert CHECKPOINT_DRIVE.is_dir(), f'{CHECKPOINT_DRIVE} not found'
assert SELECTION_SET_DRIVE.is_dir(), f'{SELECTION_SET_DRIVE} not found'
if INPUTS_LOCAL.exists():
    shutil.rmtree(INPUTS_LOCAL)
INPUTS_LOCAL.mkdir()
shutil.copytree(CHECKPOINT_DRIVE, CHECKPOINT_LOCAL)
shutil.copytree(SELECTION_SET_DRIVE, S_LOCAL)
for local in (CHECKPOINT_LOCAL, S_LOCAL):
    assert not local.resolve().is_relative_to(REPO.resolve()), local
assert_clean()
print('copied', CHECKPOINT_DRIVE, '->', CHECKPOINT_LOCAL)
print(sorted(p.name for p in CHECKPOINT_LOCAL.iterdir()))
print('copied', SELECTION_SET_DRIVE, '->', S_LOCAL)

## 10. Run the risk-v2 fit once

One call to `src.tuning.v2_risk_run` checks the copied checkpoint against the committed search evidence (hash, architecture, bundle seed), runs the provenance checks (F3 fingerprint, F2 manifest, S), reproduces M exactly on CPU, scores val-B twice (identical), fits the policy and refits it (byte-identical), and writes four files into `/content/v2_risk_out`, outside the clone. Exactly one val-B input is passed: `--cache-dir` (the `cache_a` folder of step 8) for a log-Mel winner, `--root` (the extracted FoR root) for `aasist`. There is no device argument: the runner scores on CPU. The output folder must not exist yet; on any failure the runner removes it.

Rerun rule (pre-declared for risk-v2, adapted from notebook 05): if the runner raises a check failure (search evidence, checkpoint hash, architecture, seed, provenance, M reproduction, completeness, determinism or fit), the risk-v2 run stops and the failure is reported; proceeding requires a logged decision. Only an infrastructure failure that happens before this cell starts (Drive, clone, pip, archive, copies) may be rerun from the same COMMIT. Nothing in this notebook reads or reacts to the fitted values.

In [ ]:
OUT = Path('/content/v2_risk_out')
if OUT.exists():
    raise FileExistsError(f'{OUT} already exists; the risk run runs once per runtime')
if INPUT_KIND == 'logmel':
    VAL_B_ARGS = ['--cache-dir', str(CACHE_A)]
else:
    VAL_B_ARGS = ['--root', str(AUDIO_ROOT)]
assert_clean()
stream([
    sys.executable, '-m', 'src.tuning.v2_risk_run',
    '--checkpoint', str(CHECKPOINT_LOCAL), '--selection-set', str(S_LOCAL), '--profile', str(PROFILE),
    '--config', str(REPO / 'configs/v2_candidates.yaml'), '--manifest', str(MANIFEST), *VAL_B_ARGS,
    '--out', str(OUT), '--version', VERSION,
], cwd=REPO)

## 11. Post-run identity checks

The worktree must still be clean at `COMMIT`, which shows that nothing wrote into the clone. `run_report.json` must record `COMMIT`, `VERSION`, the winner, the bundle seed and its checkpoint SHA-256 from step 5, the winner's val-B input form, CPU scoring, and the F3 profile fingerprint and config SHA-256. These are identity checks only. `diagnostics_report.json` is printed for review only: it is descriptive, and no change to the calibration, threshold or band follows from it ([docs/risk_run.md](../docs/risk_run.md) Why (e)).

In [ ]:
assert_clean()
RUN_REPORT = json.loads((OUT / 'run_report.json').read_text(encoding='utf-8'))
assert RUN_REPORT['code_commit'] == COMMIT, RUN_REPORT['code_commit']
assert RUN_REPORT['version'] == VERSION, RUN_REPORT['version']
assert RUN_REPORT['architecture'] == WINNER, RUN_REPORT['architecture']
assert RUN_REPORT['bundle_seed'] == BUNDLE_SEED, RUN_REPORT['bundle_seed']
assert RUN_REPORT['checkpoint_sha256'] == BUNDLE['checkpoint_sha256'], RUN_REPORT['checkpoint_sha256']
assert RUN_REPORT['val_b_input']['form'] == INPUT_KIND, RUN_REPORT['val_b_input']
assert RUN_REPORT['scoring']['device'] == 'cpu', RUN_REPORT['scoring']
assert RUN_REPORT['profile_fingerprint'] == F3['fingerprint'], RUN_REPORT['profile_fingerprint']
assert RUN_REPORT['profile_config_sha256'] == F3['config_sha256'], RUN_REPORT['profile_config_sha256']
print((OUT / 'diagnostics_report.json').read_text(encoding='utf-8'))

## 12. Export to Drive and write the export manifest

As notebook 05: `risk_policy.json` and `val_b_scores.npz` go to `checkpoints/risk/<VERSION>/`, and `run_report.json` and `diagnostics_report.json` to `processed/metadata/v2_risk/`. Both folders are re-checked and created with `exist_ok=False`, so nothing is overwritten. Every Drive copy is re-hashed against its `/content` source, and `v2_risk_export_manifest.json` is written next to the reports: paths relative to `DRIVE_ROOT`, sizes and SHA-256s; the commit; `VERSION`, `SEARCH_VERSION`, the winner and the bundle seed; the environment (Python, NumPy, torch, GPU name) and the `require_versions` result; UTC time; the D21 FoR revision and the observed FoR archive SHA-256 (evidence only); the pytest counts; the SHA-256 of every cache file of both builds (`null` for `aasist`); and the F3 config SHA-256 and fingerprint.

Two hash checks run. The first, right after copying, checks the copy. Drive is then flushed, unmounted and remounted, and every file is hashed again against the manifest: this second check covers what Drive actually stored after the flush, and also fails on a missing or extra file. These are candidate records until human review.

In [ ]:
from datetime import datetime, timezone

refuse_existing_outputs()
EXPORTS = {
    'risk_policy.json': CHECKPOINT_DEST,
    'val_b_scores.npz': CHECKPOINT_DEST,
    'run_report.json': EVIDENCE_DEST,
    'diagnostics_report.json': EVIDENCE_DEST,
}
assert sorted(p.name for p in OUT.iterdir()) == sorted(EXPORTS), sorted(p.name for p in OUT.iterdir())
CHECKPOINT_DEST.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DEST.mkdir(exist_ok=False)
EVIDENCE_DEST.parent.mkdir(parents=True, exist_ok=True)
EVIDENCE_DEST.mkdir(exist_ok=False)
files = []
for name, dest in EXPORTS.items():
    shutil.copy2(OUT / name, dest / name)
    digest = sha256(OUT / name)
    assert sha256(dest / name) == digest, f'{dest / name} differs from {OUT / name}'
    files.append({'path': (dest / name).relative_to(DRIVE).as_posix(), 'size': (dest / name).stat().st_size,
                  'sha256': digest})
EXPORT_MANIFEST = {
    'commit': COMMIT,
    'version': VERSION,
    'search_version': SEARCH_VERSION,
    'architecture': WINNER,
    'bundle_seed': BUNDLE_SEED,
    'environment': ENV,
    'require_versions': VERSIONS,
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'for_revision': EXPECTED_REVISION,
    'for_archive_sha256_observed': FOR_ARCHIVE_SHA256_OBSERVED,
    'pytest_counts': COUNTS,
    'cache_sha256': {'cache_a': CACHE_HASHES_A, 'cache_b': CACHE_HASHES_B},
    'f3': {'config_sha256': F3['config_sha256'], 'fingerprint': F3['fingerprint']},
    'files': files,
}
MANIFEST_PATH = EVIDENCE_DEST / 'v2_risk_export_manifest.json'
MANIFEST_TEXT = json.dumps(EXPORT_MANIFEST, indent=2, sort_keys=True) + '\n'
MANIFEST_PATH.write_text(MANIFEST_TEXT, encoding='utf-8')
print(MANIFEST_TEXT, end='')
drive.flush_and_unmount()

## 13. Remount and re-verify every exported file

Drive is remounted and every file under the two export folders is hashed again against the manifest (the manifest itself against the text just written). A missing file, an extra file or a changed byte stops the notebook.

In [ ]:
drive.mount('/content/drive', force_remount=True)
expected = {entry['path']: entry['sha256'] for entry in EXPORT_MANIFEST['files']}
expected[MANIFEST_PATH.relative_to(DRIVE).as_posix()] = hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest()
stored = {p.relative_to(DRIVE).as_posix() for dest in (CHECKPOINT_DEST, EVIDENCE_DEST)
          for p in dest.rglob('*') if p.is_file()}
missing, extra = sorted(set(expected) - stored), sorted(stored - set(expected))
assert not missing and not extra, f'missing on Drive: {missing}; extra on Drive: {extra}'
mismatched = [path for path, digest in sorted(expected.items()) if sha256(DRIVE / path) != digest]
assert not mismatched, f'SHA-256 mismatch on Drive after remount: {mismatched}'
print(f'Drive verified after remount: {len(expected)} files match the manifest.')

## 14. What is committed

After human review, the results PR commits four files under `docs/audit_evidence/v2_risk/`: `run_report.json`, `risk_policy.json` and `diagnostics_report.json` from this run, and `v2_risk_export_manifest.json`. `val_b_scores.npz` holds one line per val-B clip; it is data and stays on Drive only (`checkpoints/risk/risk-v2/`), bound to the reports by `val_b_scores_sha256` in `run_report.json` (AGENTS.md rule 7). `risk_policy.json` is the policy file F5 freezes into ModelBundle v2.

Order after this run: the results PR (the four files above) → notebook 09, the v2 Stage 11 run, which reads `docs/audit_evidence/v2_risk/` and copies `val_b_scores.npz` from `checkpoints/risk/risk-v2/` → F5.